In [1]:

import _init_paths
import torch.nn as nn
import torch

In [2]:
from frcnn_helper import *
from model.utils.config import cfg
from model.faster_rcnn.vgg16 import vgg16

In [3]:
test_model_path = "models/vgg16/KAIST/wkAVG_cr/faster_rcnn_KAIST_AVG_1.pth"

In [4]:
class Arguments():
    def __init__(self):
        self.class_agnostic =False
        self.epochs = 3
        self.lr = 0.001
        self.lr_decay_step = 4
        self.dataset = 'KAIST'
        self.net = 'vgg16'
        self.batch_size = 24
        self.nw = 2
        self.cuda = True
        #self.round = 10
        self.mGPUs=True
        self.optimizer ="sgd"
        self.k=5
        #self.device = th.device("cpu")
        
args = Arguments()

In [5]:
def initial_network(imdb_classes,args):
    
      # initilize the network here.
    if args.net == 'vgg16':
        fasterRCNN = vgg16(imdb_classes, pretrained=True, class_agnostic=args.class_agnostic)
    elif args.net == 'res101':
        fasterRCNN = resnet(imdb_classes, 101, pretrained=True, class_agnostic=args.class_agnostic)
    elif args.net == 'res50':
        fasterRCNN = resnet(imdb_classes, 50, pretrained=True, class_agnostic=args.class_agnostic)
    elif args.net == 'res152':
        fasterRCNN = resnet(imdb_classes, 152, pretrained=True, class_agnostic=args.class_agnostic)
    else:
        print("network is not defined")
        pdb.set_trace()

    fasterRCNN.create_architecture()
    


    if args.cuda:
        fasterRCNN.cuda()

    if args.mGPUs:
        fasterRCNN = nn.DataParallel(fasterRCNN)
        

        
    return fasterRCNN

In [6]:
def load_model(imdb_classes,model_path, args,cfg):
    model = initial_network(imdb_classes,args)
    checkpoint = torch.load(model_path)
    
    if args.mGPUs:
        model.module.load_state_dict(checkpoint['model'])
    else:
        model.load_state_dict(checkpoint['model'])
    
    optimizer = getOptimizer(model,args,cfg)
    optimizer.load_state_dict(checkpoint['optimizer'])
    
    start_round = checkpoint['round']
    
    return model,optimizer, start_round

In [7]:
def getOptimizer(fasterRCNN,args,cfg):
    lr = args.lr
    params = []
    for key, value in dict(fasterRCNN.named_parameters()).items():
        if value.requires_grad:
            if 'bias' in key:
                params += [{'params':[value],'lr':lr*(cfg.TRAIN.DOUBLE_BIAS + 1), \
                    'weight_decay': cfg.TRAIN.BIAS_DECAY and cfg.TRAIN.WEIGHT_DECAY or 0}]
            else:
                params += [{'params':[value],'lr':lr, 'weight_decay': cfg.TRAIN.WEIGHT_DECAY}]
                
    if args.optimizer == "adam":
        lr = lr * 0.1
        optimizer = torch.optim.Adam(params)

    elif args.optimizer == "sgd":
        optimizer = torch.optim.SGD(params, momentum=cfg.TRAIN.MOMENTUM) 
    return optimizer

In [8]:
imdb_classes =  ('__background__',  # always index 0
                          'person',
                          'people','cyclist'
                         )

In [10]:
my_model

DataParallel(
  (module): vgg16(
    (RCNN_rpn): _RPN(
      (RPN_Conv): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (RPN_cls_score): Conv2d(512, 24, kernel_size=(1, 1), stride=(1, 1))
      (RPN_bbox_pred): Conv2d(512, 48, kernel_size=(1, 1), stride=(1, 1))
      (RPN_proposal): _ProposalLayer()
      (RPN_anchor_target): _AnchorTargetLayer()
    )
    (RCNN_proposal_target): _ProposalTargetLayer()
    (RCNN_roi_pool): ROIPool(output_size=(7, 7), spatial_scale=0.0625)
    (RCNN_roi_align): ROIAlign(output_size=(7, 7), spatial_scale=0.0625, sampling_ratio=0)
    (RCNN_base): Sequential(
      (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace)
      (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU(inplace)
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (6)

In [9]:
my_model,optimizer, start_round = load_model(imdb_classes,test_model_path, args,cfg)

Loading pretrained weights from data/pretrained_model/vgg16_caffe.pth


In [4]:
#testimg_pickle_path = 'testimg2252.pkl'
import pickle
testimg_pickle_path = 'data/pickle/MI3_test_3325.pkl'
with open(testimg_pickle_path, 'rb') as handle:
    test_images = pickle.load(handle)

In [6]:
len(test_images)

3225

In [8]:
imgs = test_images
batch_sz=64
n_batch = len(imgs)//batch_sz
res_batch = len(imgs)%batch_sz    

In [10]:
n_batch

50

In [11]:
    with torch.no_grad():
        for i in range(n_batch):
            in_batch = torch.cat(imgs[i*batch_sz:(i+1)*batch_sz]).to('cuda')
#             x = rcnn_intermediate(frcnn_model, in_batch)
#             in_batch.to('cpu')
#             outputs.append(x)

#         # last incomplete batch.
#         in_batch = torch.cat(imgs[-res_batch:]).to('cuda')

In [18]:
in_batch[0].size()


torch.Size([3, 640, 480])

In [5]:
test_images

tensor([[[[-102.9801, -102.9801, -102.9801,  ...,  -99.9801, -102.9801,
           -102.9801],
          [-102.9801, -102.9801, -102.9801,  ...,  -99.9801, -102.9801,
           -102.9801],
          [-102.9801, -102.9801, -102.9801,  ...,  -99.9801, -102.9801,
           -102.9801],
          ...,
          [-102.9801, -102.9801, -102.9801,  ...,  -99.9801, -102.9801,
           -102.9801],
          [-102.9801, -102.9801, -102.9801,  ...,  -99.9801, -102.9801,
           -102.9801],
          [-102.9801, -102.9801, -102.9801,  ...,  -99.9801, -102.9801,
           -102.9801]],

         [[-115.9465, -115.9465, -115.9465,  ..., -112.9465, -115.9465,
           -115.9465],
          [-115.9465, -115.9465, -115.9465,  ..., -112.9465, -115.9465,
           -115.9465],
          [-115.9465, -115.9465, -115.9465,  ..., -112.9465, -115.9465,
           -115.9465],
          ...,
          [-115.9465, -115.9465, -115.9465,  ..., -112.9465, -115.9465,
           -115.9465],
          [-115.94

In [ ]:
if args.mGPUs:
    fasterRCNN = my_model.module
X = get_features(fasterRCNN, test_images, args.batch_size)/255.0


In [ ]:
wk_value = within_cluster_dispersion(X, n_cluster=args.k)
wk_list.append(wk_value)
print(wk_value)